# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Note: metadata is an object, not a dictionary
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

# Optionally, let's pretty-print the main metadata fields
print("\nAdditional Metadata:")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Cite as: {dataset.metadata.citeAs}")

## 2. Data Overview
List available record sets, fields, columns, and their `@id`s. Useful for referencing data in downstream operations.

In [ ]:
# List all record sets and their fields by their @id
print("RecordSets available in this dataset:")
record_sets = list(dataset.metadata.recordSets)
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs['name']}")
    print(f"  Description: {rs.get('description', '')}")
    print(f"  Fields:")
    for field in rs['fields']:
        print(f"    Field @id: {field['@id']} | Name: {field.get('name', '')}")
    # If columns are defined (for tabular data)
    if 'columns' in rs:
        print(f"  Columns:")
        for col in rs['columns']:
            print(f"    Column @id: {col['@id']} | Name: {col.get('name', '')}")

## 3. Data Extraction
Load data from a specific RecordSet into a DataFrame for analysis. All record sets are referenced by their `@id`s.

In [ ]:
# Gather all recordSet @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
dataframes = {}

# Extract data for each RecordSet
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet: {record_set_id} (Columns: {df.columns.tolist()})\n")

# For demonstration, preview one dataframe (if available)
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"First 5 records of RecordSet {example_rs_id}:")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering, normalization, and grouping.

- Pick a numeric field (`@id`) and group field (`@id`) from the RecordSet overview above.

In [ ]:
# Please adjust the following IDs to match your dataset's fields and columns
# Example values used for demonstration. Replace them with actual @id's for your analysis
rs_id = record_set_ids[0]  # Use the first RecordSet
df = dataframes[rs_id]

# List columns so the user knows what to refer to
print("DataFrame columns (use @id fields):")
print(df.columns)

# Choose a numeric field and a group field from the above columns
numeric_field = None
group_field = None
# Simple heuristic: try to find a likely numeric field (e.g., 'MW', 'abundance', etc.)
for col in df.columns:
    if ('abundance' in col.lower()) or ('coverage' in col.lower()) or ('count' in col.lower()) or ('mw' in col.lower()):
        numeric_field = col
        break
for col in df.columns:
    if ('group' in col.lower()) or ('sample' in col.lower()) or ('category' in col.lower()):
        group_field = col
        break

# Fallback
if numeric_field is None and len(df.columns) > 0:
    numeric_field = df.columns[0]
if group_field is None and len(df.columns) > 1:
    group_field = df.columns[1]

print(f"Numeric field \"@id\": {numeric_field}")
print(f"Group field \"@id\": {group_field}")

# Convert numeric field to float if possible
if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]) == False:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter records where numeric_field > threshold
threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fiu' else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize numeric_field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field if present
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field or the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

if group_field and group_field in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} across {group_field} groups")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a biological mass spectrometry dataset described by a Croissant schema using `mlcroissant`, explored its structure by referencing all entities by their `@id`s, extracted tabular data, performed pre-processing and EDA (including normalization and grouping), and visualized the results.

With these steps, you can begin more advanced machine learning or scientific analyses of the mass spectrometry data on extracellular vesicles from human mast cells.